# Figure S6 — inter-session reproducibility of the standard synthesis (flows 2.8 × 5 µL/s)

Original pipeline `paperAnalysis/analyse_reproducibility/reproducibility_pipeline.py`, vendored verbatim in `reproducibility_pipeline.py` (only DATA_DIR = raw/ and OUT_DIR changed), run as its own entry point by `toofast_analysis.figS06`. Outputs → `figures/S06/`.

Selection: every standard sample in raw/ (H5 + legacy CSV), manips with ≥ 2 samples whose final spectra match (RMSE < 0.02, |Δλ| < 3 nm), then Ext_max ≥ 0.10 and 500 ≤ λmax ≤ 600 nm; λmax = parabolic vertex after bridging the 510–535 nm lamp notch.

⚑ **Not reproduced**: the paper quotes n = 245, 547 nm, 21.3 nm, CV 3.9 %. The code gives 117 manips, 548.2 nm, 21.9 nm, 4.0 % on raw/ (the original outputs, 119 / 548.3 / 21.7 / 3.96 %, counted two duplicate folders twice). Lead: 246 stable *samples* (spectra) in the filtered set — the caption speaks of 245 spectra.

## Analysis code
All the code of this figure is in this notebook.

### `paths`

In [ ]:
import sys
import types

from IPython.core.magic import register_cell_magic

MODULE_SOURCE = {}


@register_cell_magic
def module(line, cell):
    """`%%module a.b`: run this cell as the Python module a.b, so that the original code keeps its own namespaces."""
    name = line.strip()
    parts = name.split(".")
    for i in range(1, len(parts)):                              # parent packages
        pkg = ".".join(parts[:i])
        if pkg not in sys.modules:
            sys.modules[pkg] = types.ModuleType(pkg)
            sys.modules[pkg].__path__ = []
            if i > 1:
                setattr(sys.modules[".".join(parts[:i - 1])], parts[i - 1], sys.modules[pkg])
    mod = types.ModuleType(name)
    mod.__package__ = name.rpartition(".")[0]
    sys.modules[name] = mod
    setattr(sys.modules[mod.__package__], parts[-1], mod)
    MODULE_SOURCE[name] = cell
    exec(compile(cell, f"<{name}>", "exec"), mod.__dict__)


In [ ]:
%%module toofast_analysis.paths
"""Repository paths."""
import os
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment.yml").exists())   # notebook: no __file__
# raw/ can live elsewhere (e.g. an extracted Zenodo archive): set TOOFAST_RAW
RAW = Path(os.environ.get("TOOFAST_RAW", ROOT / "raw"))
PROCESSED = ROOT / "data" / "processed"
INPUTS = ROOT / "data" / "inputs"      # frozen artefacts used by the platform during the runs (versioned, read-only)
RESOURCE_DIR = INPUTS
FIGURES = ROOT / "figures"


In [ ]:
from toofast_analysis.paths import *

### `reproducibility_pipeline`
*Original analysis code.*

In [ ]:
%%module toofast_analysis.reproducibility_pipeline
# [vendored] verbatim copy of paperAnalysis/analyse_reproducibility/reproducibility_pipeline.py (md5 270a3fccad069da2b334c0fef10b3189), tools/vendor_s6.py — paths only.
"""
Engine script for the interexperiment reproducibility analysis (flow = 2.8x5).

Validated standalone (`py -3 reproducibility_pipeline.py`) before being ported
into `reproducibility_analysis.ipynb`. Kept alongside the notebook so the same
logic can be re-run headless.
"""
import itertools
from collections import defaultdict
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter

# ============================================================================
# CONFIGURATION
# ============================================================================
from toofast_analysis.paths import RAW as _RAW, FIGURES as _FIG   # [vendored]
DATA_DIR = _RAW
OUT_DIR = _FIG / "S06"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_FLOWS = np.array([2.8, 2.8, 2.8, 2.8, 2.8])
FLOW_TOL = 0.05          # mL/min tolerance to call a manip "flow = 2.8x5"

WL_MIN, WL_MAX, N_POINTS = 315, 750, 500   # harmonization grid
# the "final" spectrum of a sample is its last acquired spectrum (smoothed,
# not averaged with earlier points), i.e. the closest snapshot to the true
# end-of-reaction / steady-state state.

# "Two samples have the same final spectrum" criterion:
RMSE_THRESHOLD = 0.02     # max RMSE (absorbance units) between two harmonized final spectra
LAMBDA_TOL_NM = 3.0       # max |delta lambda_max| between two samples

# calibrated against the actual dataset (see notebook section 3): with these
# thresholds ~20% of same-manip sample pairs are called "equal", giving ~165
# stable manips across ~45 campaigns/chips out of 575 manips with >=2 samples.

# Quality filters applied AFTER stability selection:
# - some manips are reproducibly flat (both/all matching samples failed to
#   react -> near-zero, featureless absorbance). These pass the equality test
#   trivially but are not a real plasmon peak and would inflate lambda_max
#   variance artificially.
ABS_MAX_MIN = 0.10
# - manips whose peak falls outside the expected AgNP plasmon band are not
#   comparable syntheses (different/failed chemistry, stray artifact, ...)
#   and are excluded from the interexperiment comparison.
LAMBDA_MAX_MIN_NM = 500.0
LAMBDA_MAX_MAX_NM = 600.0

# Lamp-crossover artifact: every spectrum (both spectrometer models) shows a
# shallow (~1-4% relative), narrow notch around 510-535 nm, at essentially the
# same wavelength regardless of the true peak position. It is consistent with
# a source-mixing crossover of the combined deuterium/halogen lamp. Because
# the true plasmon band is broad and flat in this region, the notch creates
# two competing local maxima and argmax "hops" between them from one manip to
# the next, producing an artificial gap in the lambda_max distribution around
# 510-535 nm. See notebook section 3 for the empirical evidence.
NOTCH_BAND_NM = (510, 535)   # nm, region bridged out before peak detection
NOTCH_ANCHOR_NM = 15         # nm of spectrum on each side used to anchor the bridge
PEAK_FIT_WINDOW_NM = 40      # nm half-window for the parabolic vertex fit


# ============================================================================
# LOADING
# ============================================================================
def _smooth(spectrum, n_pixels):
    s = spectrum.copy()
    if n_pixels == 3198:
        # dead-pixel correction for this spectrometer model
        s[1200:1210] = s[1199]
        s[1900:1910] = s[1899]
        s[2460:2472] = s[2459]
        s = savgol_filter(s, window_length=101, polyorder=3)
    elif n_pixels == 1219:
        s = savgol_filter(s, window_length=51, polyorder=3)
    else:
        w = min(51, n_pixels // 10)
        if w % 2 == 0:
            w += 1
        s = savgol_filter(s, window_length=w, polyorder=3)
    return s


def load_sample(h5_path):
    """Load one sample H5 file. Returns None if flow doesn't match target or format unknown."""
    with h5py.File(h5_path, "r") as f:
        if "conditions/flow_set" not in f:
            return None
        flows = np.array(f["conditions/flow_set"][:])
        wl = np.array(f["spectra/wavelength"][:])
        abso = np.array(f["spectra/extinction"][:])

        if flows.shape[0] != 5 or not np.all(np.abs(flows - TARGET_FLOWS) < FLOW_TOL):
            return None

        if abso.ndim == 1:
            abso = abso.reshape(1, -1)

        meta = f["sample"].attrs
        n_pixels = len(wl)
        final_mean = _smooth(abso[-1], n_pixels)

        return dict(
            path=h5_path,
            manip_dir=h5_path.parent.parent,
            campaign=h5_path.parent.parent.parent.name,
            manip=h5_path.parent.parent.name,
            exp_name=str(meta.get("run", h5_path.parent.parent.name)),
            sample_id=str(meta.get("sample_id", h5_path.stem)),
            chip_id=str(meta.get("chip_id", "")),
            timestamp=str(meta.get("timestamp", "")),
            wl=wl,
            final_mean=final_mean,
            n_spectra=abso.shape[0],
            n_pixels=n_pixels,
        )


def load_sample_csv(csv_path, manip_dir, sample_idx):
    """Load one legacy-format 'measurement_spectra_sample_N.csv' file (older
    campaigns, pre-H5 acquisition pipeline: no 'samples/' subfolder, spectra
    stored directly as semicolon-separated CSV next to pump_flows.csv)."""
    df = pd.read_csv(csv_path, sep=";")
    wl_cols = df.columns[1:]
    wl = np.array([float(c) for c in wl_cols])
    abso = df[wl_cols].to_numpy(dtype=float)
    if abso.shape[0] == 0:
        return None

    n_pixels = len(wl)
    final_mean = _smooth(abso[-1], n_pixels)

    return dict(
        path=csv_path,
        manip_dir=manip_dir,
        campaign=manip_dir.parent.name,
        manip=manip_dir.name,
        exp_name=manip_dir.name,
        sample_id=f"sample_{sample_idx}",
        chip_id="",
        timestamp=str(df.iloc[0, 0]) if len(df) else "",
        wl=wl,
        final_mean=final_mean,
        n_spectra=abso.shape[0],
        n_pixels=n_pixels,
    )


def find_csv_samples(data_dir, exclude_tops):
    """Recursively find every legacy-CSV sample with flow ~= TARGET_FLOWS,
    skipping any top-level campaign folder already covered by the H5 scan
    (`exclude_tops`) to avoid double-counting manips that have both formats."""
    samples = []
    for pf in data_dir.rglob("pump_flows.csv"):
        top = pf.relative_to(data_dir).parts[0]
        if top in exclude_tops:
            continue
        manip_dir = pf.parent
        try:
            rows = np.array(
                [line.split(";") for line in pf.read_text().strip().splitlines() if line.strip()],
                dtype=float,
            )
        except Exception:
            continue
        if rows.ndim != 2 or rows.shape[1] != 5:
            continue

        for csv_path in sorted(manip_dir.glob("measurement_spectra_sample_*.csv")):
            try:
                idx = int(csv_path.stem.rsplit("_", 1)[-1])
            except ValueError:
                continue
            if idx - 1 >= len(rows):
                continue
            if not np.all(np.abs(rows[idx - 1] - TARGET_FLOWS) < FLOW_TOL):
                continue
            try:
                s = load_sample_csv(csv_path, manip_dir, idx)
            except Exception:
                s = None
            if s is not None:
                samples.append(s)
    return samples


def find_standard_flow_samples(data_dir):
    """Find every sample with flow ~= TARGET_FLOWS, from both acquisition
    formats found in the data folder: the newer per-sample H5 files
    (`.../<manip>/samples/*.h5`, at any nesting depth) and the older
    semicolon-CSV format (`.../<manip>/measurement_spectra_sample_N.csv` +
    `pump_flows.csv`, no 'samples' subfolder) used before the H5 pipeline."""
    h5_files = sorted(set(data_dir.rglob("samples/*.h5")))
    samples = []
    for h5p in h5_files:
        try:
            s = load_sample(h5p)
        except Exception:
            s = None
        if s is not None:
            samples.append(s)

    h5_tops = set(p.relative_to(data_dir).parts[0] for p in h5_files)
    samples += find_csv_samples(data_dir, exclude_tops=h5_tops)
    return samples


def harmonize(wl, spectrum, wl_min=WL_MIN, wl_max=WL_MAX, n_points=N_POINTS):
    target_wl = np.linspace(wl_min, wl_max, n_points)
    mask = (wl >= wl_min) & (wl <= wl_max)
    interp = interp1d(wl[mask], spectrum[mask], kind="linear", bounds_error=False, fill_value=0.0)
    return target_wl, interp(target_wl)


# ============================================================================
# LAMP-ARTIFACT CORRECTION + ROBUST PEAK DETECTION
# ============================================================================
def correct_notch(wl, spectrum, band=NOTCH_BAND_NM, anchor_nm=NOTCH_ANCHOR_NM):
    """Bridge the lamp-crossover notch with a straight line anchored on the
    (noise-averaged) spectrum just outside the band, so the peak underneath
    is not artificially split in two."""
    band_mask = (wl >= band[0]) & (wl <= band[1])
    if not band_mask.any():
        return spectrum
    dwl = wl[1] - wl[0]
    n_anchor = max(1, int(round(anchor_nm / dwl)))
    i0, i1 = np.where(band_mask)[0][[0, -1]]
    left_val = spectrum[max(0, i0 - n_anchor):i0].mean() if i0 > 0 else spectrum[i0]
    right_val = spectrum[i1 + 1:i1 + 1 + n_anchor].mean() if i1 + 1 < len(spectrum) else spectrum[i1]
    left_wl = wl[max(0, i0 - n_anchor // 2)]
    right_wl = wl[min(len(wl) - 1, i1 + 1 + n_anchor // 2)]
    out = spectrum.copy()
    out[i0:i1 + 1] = np.interp(wl[i0:i1 + 1], [left_wl, right_wl], [left_val, right_val])
    return out


def robust_lambda_max(wl, spectrum, window_nm=PEAK_FIT_WINDOW_NM):
    """Parabolic vertex fit around the global max of the (notch-corrected)
    spectrum. More robust than plain argmax against residual instrumental
    ripple: it reports the vertex of the local quadratic envelope rather than
    the single noisiest/highest pixel."""
    idx = np.argmax(spectrum)
    dwl = wl[1] - wl[0]
    half_w = int(round(window_nm / dwl))
    lo, hi = max(0, idx - half_w), min(len(spectrum), idx + half_w + 1)
    x, y = wl[lo:hi], spectrum[lo:hi]
    coeffs = np.polyfit(x, y, 2)
    if coeffs[0] >= 0:   # not concave -> fall back to argmax
        return wl[idx], spectrum[idx]
    vertex_wl = -coeffs[1] / (2 * coeffs[0])
    if not (wl[0] <= vertex_wl <= wl[-1]):
        return wl[idx], spectrum[idx]
    vertex_abs = np.polyval(coeffs, vertex_wl)
    return vertex_wl, vertex_abs


# ============================================================================
# SPECTRAL DESCRIPTORS
# ============================================================================
def fwhm_of(wl, spectrum):
    """Full width at half max around the global peak, via linear interpolation
    of the half-max crossings. Returns np.nan if the peak is not bracketed."""
    idx = np.argmax(spectrum)
    half = spectrum[idx] / 2.0
    left = np.nan
    for i in range(idx, 0, -1):
        if spectrum[i] >= half >= spectrum[i - 1]:
            left = np.interp(half, [spectrum[i - 1], spectrum[i]], [wl[i - 1], wl[i]])
            break
    right = np.nan
    for i in range(idx, len(spectrum) - 1):
        if spectrum[i] >= half >= spectrum[i + 1]:
            right = np.interp(half, [spectrum[i + 1], spectrum[i]], [wl[i + 1], wl[i]])
            break
    if np.isnan(left) or np.isnan(right):
        return np.nan
    return right - left


def auc_of(wl, spectrum):
    trapezoid = getattr(np, "trapezoid", np.trapz)
    return trapezoid(spectrum, wl)


# ============================================================================
# STABILITY ANALYSIS (within-manip agreement across samples)
# ============================================================================
def group_by_manip(samples):
    manips = defaultdict(list)
    for s in samples:
        manips[s["manip_dir"]].append(s)
    return manips


def connected_components(n, edges):
    """Simple union-find-free BFS grouping for small n (2-5 samples/manip)."""
    adj = defaultdict(set)
    for i, j in edges:
        adj[i].add(j)
        adj[j].add(i)
    seen = set()
    components = []
    for start in range(n):
        if start in seen or start not in adj:
            continue
        stack, comp = [start], set()
        while stack:
            node = stack.pop()
            if node in comp:
                continue
            comp.add(node)
            seen.add(node)
            stack.extend(adj[node] - comp)
        components.append(comp)
    return components


def analyze_manip(manip_dir, sample_list):
    """Return a result dict for one manip, or None if <2 samples or no stable pair."""
    if len(sample_list) < 2:
        return None

    harmonized = []
    lambdas = []
    for s in sample_list:
        hwl, hspec = harmonize(s["wl"], s["final_mean"])
        hspec = correct_notch(hwl, hspec)
        lm, _ = robust_lambda_max(hwl, hspec)
        harmonized.append(hspec)
        lambdas.append(lm)

    n = len(sample_list)
    equal_edges = []
    for i, j in itertools.combinations(range(n), 2):
        rmse = np.sqrt(np.mean((harmonized[i] - harmonized[j]) ** 2))
        if rmse <= RMSE_THRESHOLD and abs(lambdas[i] - lambdas[j]) <= LAMBDA_TOL_NM:
            equal_edges.append((i, j))

    if not equal_edges:
        return None

    components = connected_components(n, equal_edges)
    best = max(components, key=len)
    idx = sorted(best)

    hwl = np.linspace(WL_MIN, WL_MAX, N_POINTS)
    group_spectra = np.array([harmonized[i] for i in idx])
    rep_spectrum = group_spectra.mean(axis=0)
    intra_std_spectrum = group_spectra.std(axis=0)

    rep_lambda, rep_abs = robust_lambda_max(hwl, rep_spectrum)
    fwhm = fwhm_of(hwl, rep_spectrum)
    auc = auc_of(hwl, rep_spectrum)
    group_lambdas = [lambdas[i] for i in idx]

    return dict(
        campaign=sample_list[0]["campaign"],
        manip=sample_list[0]["manip"],
        manip_dir=str(manip_dir),
        chip_id=sample_list[0]["chip_id"],
        timestamp=sample_list[0]["timestamp"],
        n_samples_total=n,
        n_samples_stable=len(idx),
        matched_sample_ids=[sample_list[i]["sample_id"] for i in idx],
        wl=hwl,
        rep_spectrum=rep_spectrum,
        intra_std_spectrum=intra_std_spectrum,
        lambda_max=rep_lambda,
        abs_max=rep_abs,
        fwhm=fwhm,
        auc=auc,
        intra_lambda_spread=float(np.ptp(group_lambdas)),
        intra_rmse_max=float(max(
            np.sqrt(np.mean((group_spectra[a] - group_spectra[b]) ** 2))
            for a, b in itertools.combinations(range(len(idx)), 2)
        )) if len(idx) > 1 else 0.0,
    )


# ============================================================================
# MAIN PIPELINE
# ============================================================================
def run():
    print(f"Scanning {DATA_DIR} for sample H5 files...")
    samples = find_standard_flow_samples(DATA_DIR)
    print(f"  {len(samples)} samples with flow ~= {list(TARGET_FLOWS)} (tol={FLOW_TOL})")

    manips = group_by_manip(samples)
    print(f"  {len(manips)} distinct manips represented")

    results = []
    for manip_dir, slist in manips.items():
        r = analyze_manip(manip_dir, slist)
        if r is not None:
            results.append(r)

    print(f"  {len(results)} STABLE manips (>=2 samples with matching final spectrum)")
    campaigns = sorted(set(r["campaign"] for r in results))
    print(f"  {len(campaigns)} distinct campaigns/chips represented among stable manips")

    return samples, manips, results


def results_to_df(results):
    return pd.DataFrame([{
        "campaign": r["campaign"], "manip": r["manip"], "chip_id": r["chip_id"],
        "timestamp": r["timestamp"], "n_samples_total": r["n_samples_total"],
        "n_samples_stable": r["n_samples_stable"],
        "matched_sample_ids": ",".join(r["matched_sample_ids"]),
        "lambda_max_nm": r["lambda_max"], "abs_max": r["abs_max"],
        "fwhm_nm": r["fwhm"], "auc": r["auc"],
        "intra_lambda_spread_nm": r["intra_lambda_spread"],
        "intra_rmse_max": r["intra_rmse_max"],
        "manip_dir": r["manip_dir"],
    } for r in results]).sort_values(["campaign", "manip"]).reset_index(drop=True)


def summarize(df, cols=("lambda_max_nm", "abs_max", "fwhm_nm", "auc")):
    rows = []
    for col in cols:
        vals = df[col].dropna().values
        rows.append(dict(
            descriptor=col, n=len(vals), mean=vals.mean(), std=vals.std(ddof=1),
            median=np.median(vals), cv_pct=vals.std(ddof=1) / vals.mean() * 100,
            min=vals.min(), max=vals.max(),
        ))
    return pd.DataFrame(rows)


def variance_decomposition(df, col, group_col="campaign"):
    """One-way random-effects ANOVA decomposition of `col` into a within-group
    (same campaign/chip folder, different manip subfolders -> intra-session)
    and a between-group (different campaign folder -> inter-session,
    different chip/reagent batch) variance component.

    Uses the standard method-of-moments estimator for unbalanced designs
    (Searle et al.). Returns np.nan for var_between if it comes out negative
    (i.e. no detectable between-group effect beyond within-group noise).
    """
    groups = [g[col].dropna().values for _, g in df.groupby(group_col)]
    groups = [g for g in groups if len(g) >= 1]
    k = len(groups)
    n_total = sum(len(g) for g in groups)
    grand_mean = np.concatenate(groups).mean()

    ss_within = sum(((g - g.mean()) ** 2).sum() for g in groups)
    df_within = n_total - k
    ms_within = ss_within / df_within if df_within > 0 else np.nan

    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
    df_between = k - 1
    ms_between = ss_between / df_between

    ns = np.array([len(g) for g in groups])
    n0 = (n_total - (ns ** 2).sum() / n_total) / (k - 1)

    var_within = ms_within
    var_between = (ms_between - ms_within) / n0 if not np.isnan(ms_within) else np.nan
    var_between = max(0.0, var_between) if not np.isnan(var_between) else np.nan

    sd_within = np.sqrt(var_within) if not np.isnan(var_within) else np.nan
    sd_between = np.sqrt(var_between) if not np.isnan(var_between) else np.nan
    icc = (var_between / (var_between + var_within)
           if not (np.isnan(var_between) or np.isnan(var_within)) else np.nan)

    return dict(
        descriptor=col, n_manips=n_total, n_campaigns=k,
        n_campaigns_multi=int((ns >= 2).sum()),
        sd_within=sd_within, sd_between=sd_between,
        sd_total=np.sqrt(np.nansum([var_within, var_between])),
        icc_between_pct=icc * 100 if not np.isnan(icc) else np.nan,
    )


PUB_STYLE = {
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.linewidth": 1.1,
    "axes.edgecolor": "black",
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "xtick.top": True,
    "ytick.right": True,
    "legend.frameon": False,
    "figure.dpi": 130,
    "savefig.dpi": 300,
}


if __name__ == "__main__":
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update(PUB_STYLE)

    samples, manips, results = run()
    df_all = results_to_df(results)
    out_csv = OUT_DIR / "stable_manips_flows_2.8.csv"
    df_all.to_csv(out_csv, index=False)
    print(f"\nSaved per-manip results (all stable manips): {out_csv}")

    quality_mask = (
        (df_all["abs_max"] >= ABS_MAX_MIN)
        & (df_all["lambda_max_nm"] >= LAMBDA_MAX_MIN_NM)
        & (df_all["lambda_max_nm"] <= LAMBDA_MAX_MAX_NM)
    )
    df = df_all[quality_mask].reset_index(drop=True)
    print(f"Quality filter abs_max >= {ABS_MAX_MIN} and "
          f"{LAMBDA_MAX_MIN_NM} <= lambda_max <= {LAMBDA_MAX_MAX_NM} nm: "
          f"{len(df)}/{len(df_all)} manips kept "
          f"({df_all['campaign'].nunique()} -> {df['campaign'].nunique()} campaigns)")

    stats_df = summarize(df)
    stats_csv = OUT_DIR / "reproducibility_stats_flows_2.8.csv"
    stats_df.to_csv(stats_csv, index=False)
    print(f"Saved descriptor statistics: {stats_csv}\n")
    print(stats_df.to_string(index=False))

    # ---- mean spectrum +/- interexperiment std, across quality-filtered stable manips ----
    id_to_spectrum = {(r["campaign"], r["manip"]): r["rep_spectrum"] for r in results}
    wl = next(r["wl"] for r in results)
    spectra_stack = np.array([id_to_spectrum[(c, m)] for c, m in zip(df["campaign"], df["manip"])])
    mean_spec = spectra_stack.mean(axis=0)
    std_spec = spectra_stack.std(axis=0, ddof=1)

    C_MAIN, C_FILL, C_HIST, C_MEAN = "#2166ac", "#92c5de", "#4393c3", "#b2182b"

    fig, axes = plt.subplots(1, 2, figsize=(9.5, 4))
    ax = axes[0]
    for spec in spectra_stack:
        ax.plot(wl, spec, color="0.75", alpha=0.3, linewidth=0.5)
    ax.plot(wl, mean_spec, color=C_MAIN, linewidth=1.6, label="Mean")
    ax.fill_between(wl, mean_spec - std_spec, mean_spec + std_spec, color=C_FILL,
                     alpha=0.5, linewidth=0, label=r"Mean $\pm$ 1$\sigma$")
    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Absorbance (a.u.)")
    ax.set_title(f"n = {len(df)} manips, {df['campaign'].nunique()} chips", fontsize=9)
    ax.legend(fontsize=8)

    ax = axes[1]
    lm = df["lambda_max_nm"].values
    bin_w = 10.0
    lm_bins = np.arange(np.floor(lm.min() / bin_w) * bin_w, np.ceil(lm.max() / bin_w) * bin_w + bin_w, bin_w)
    ax.hist(lm, bins=lm_bins, color=C_HIST, alpha=0.75, edgecolor="white")
    ax.axvspan(*NOTCH_BAND_NM, color="0.4", alpha=0.12, label="Lamp-notch band (residual)")
    ax.axvline(lm.mean(), color=C_MEAN, linestyle="--", linewidth=1.3, label=f"Mean = {lm.mean():.1f} nm")
    ax.axvspan(lm.mean() - lm.std(ddof=1), lm.mean() + lm.std(ddof=1), color=C_MEAN, alpha=0.1)
    ax.set_xlabel(r"$\lambda_{max}$ (nm)")
    ax.set_ylabel("Number of experiments")
    ax.set_title(f"Mean = {lm.mean():.1f} $\\pm$ {lm.std(ddof=1):.1f} nm (CV = {lm.std(ddof=1)/lm.mean()*100:.1f}%)",
                 fontsize=9)
    ax.legend(fontsize=8)

    plt.tight_layout()
    fig_path = OUT_DIR / "reproducibility_interexperiment.png"
    plt.savefig(fig_path, bbox_inches="tight", facecolor="white")
    plt.savefig(OUT_DIR / "reproducibility_interexperiment.pdf", bbox_inches="tight", facecolor="white")
    print(f"\nSaved figure: {fig_path}")

    # ---- residual-dip check: is the under-density near 525-535 nm a correction
    #      leftover (lamp notch, section on NOTCH_BAND_NM) or real chemistry?
    #      Both series computed from the SAME (unfiltered-by-lambda) manip set,
    #      so they're directly comparable in scale (matches notebook section 3b). ----
    target_wl = np.linspace(WL_MIN, WL_MAX, N_POINTS)
    lm_before_all, lm_after_all = [], []
    for manip_dir, slist in manips.items():
        if len(slist) < 2:
            continue
        harm = [harmonize(s["wl"], s["final_mean"])[1] for s in slist]
        rep = np.mean(harm, axis=0)
        if rep.max() < ABS_MAX_MIN:
            continue
        lm_before_all.append(target_wl[np.argmax(rep)])
        lm_after_all.append(robust_lambda_max(target_wl, correct_notch(target_wl, rep))[0])
    lm_before_all = np.array(lm_before_all)
    lm_after_all = np.array(lm_after_all)

    bin_edges = np.arange(500, 601, 5)
    count_before, _ = np.histogram(lm_before_all, bins=bin_edges)
    count_after, _ = np.histogram(lm_after_all, bins=bin_edges)
    print("\nResidual-dip check (raw argmax vs corrected lambda_max, same manip set, 5 nm bins):")
    header = "bin".ljust(12) + "before".rjust(8) + "after".rjust(8)
    print(header)
    for e, b, a in zip(bin_edges, count_before, count_after):
        label = f"{int(e)}-{int(e + 5)}".ljust(12)
        print(label + str(b).rjust(8) + str(a).rjust(8))
    print(f"NOTCH_BAND_NM = {NOTCH_BAND_NM}: raw argmax is essentially absent from this band "
          f"(instrumental artifact); the correction fills most of it back in, leaving only a "
          f"smaller residual dip at its trailing edge - not real chemistry (see notebook section 5b "
          f"for the campaign-diversity check).")

    # ---- within-session (same chip folder) vs between-session variance decomposition ----
    decomp_df = pd.DataFrame([variance_decomposition(df, col) for col in
                               ("lambda_max_nm", "abs_max", "fwhm_nm", "auc")])
    decomp_csv = OUT_DIR / "variance_decomposition_flows_2.8.csv"
    decomp_df.to_csv(decomp_csv, index=False)
    print(f"\nWithin-session vs between-session variance decomposition ({decomp_df['n_campaigns_multi'].iloc[0]}/"
          f"{decomp_df['n_campaigns'].iloc[0]} campaigns have >=2 stable manips):\n")
    print(decomp_df.to_string(index=False))
    print(f"Saved: {decomp_csv}")

    # ---- lambda_max per campaign, sorted by campaign mean ----
    order = df.groupby("campaign")["lambda_max_nm"].mean().sort_values().index
    fig, ax = plt.subplots(figsize=(11, 4))
    for i, camp in enumerate(order):
        vals = df.loc[df["campaign"] == camp, "lambda_max_nm"].values
        ax.scatter([i] * len(vals), vals, color=C_MAIN, s=16, alpha=0.85, zorder=3)
        if len(vals) > 1:
            ax.plot([i, i], [vals.min(), vals.max()], color="0.6", linewidth=1, zorder=2)
    ax.axhline(df["lambda_max_nm"].mean(), color=C_MEAN, linestyle="--", linewidth=1,
               label="Grand mean")
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, rotation=90, fontsize=6)
    ax.set_ylabel(r"$\lambda_{max}$ (nm)")
    ax.set_xlabel("Campaign / chip")
    row = decomp_df[decomp_df["descriptor"] == "lambda_max_nm"].iloc[0]
    ax.set_title(rf"Within-chip $\sigma$ = {row.sd_within:.1f} nm   |   "
                 rf"Between-chip $\sigma$ = {row.sd_between:.1f} nm   |   "
                 rf"ICC = {row.icc_between_pct:.0f}%", fontsize=9)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "lambda_max_by_campaign.png", bbox_inches="tight", facecolor="white")
    print(f"Saved figure: {OUT_DIR / 'lambda_max_by_campaign.png'}")


### `figS06`

In [ ]:
"""Figure S6 — inter-session reproducibility of the standard synthesis (flows 2.8 x 5 uL/s).

Runs the original pipeline (paperAnalysis/analyse_reproducibility/reproducibility_pipeline.py, vendored verbatim in
toofast_analysis.reproducibility_pipeline, DATA_DIR = raw/) exactly as its `python reproducibility_pipeline.py` entry point:
scan every standard sample, keep manips with >= 2 samples whose final spectra match (RMSE < 0.02, |dlambda| < 3 nm),
quality filter Ext_max >= 0.10 and 500 <= lambda_max <= 600 nm. Outputs -> figures/S06/
(reproducibility_interexperiment.{png,pdf} = S6, stable_manips / stats / variance_decomposition CSV).

On raw/ the two unsuffixed duplicate folders S53-M01 and S54-M01 (identical to their _STD copies, removed from raw/
by the "keep the suffixed copy" rule) are no longer counted twice: 117 manips (original outputs: 119).
"""


OUT = FIGURES / "S06"


def main():
    import matplotlib
    matplotlib.use("Agg")
    ns = {"__name__": "__main__", "__package__": "toofast_analysis"}       # the pipeline, as its script entry point
    with matplotlib.rc_context():
        exec(compile(MODULE_SOURCE["toofast_analysis.reproducibility_pipeline"], "reproducibility_pipeline", "exec"), ns)
    return ns


def summary(ns):
    """Counts and statistics quoted in the paper, from the pipeline namespace."""
    df_all, df = ns["df_all"], ns["df"]
    lm = df["lambda_max_nm"]
    return dict(n_samples_scanned=len(ns["samples"]), n_manips_with_samples=len(ns["manips"]),
                n_stable_manips=len(df_all), n_manips=len(df), n_campaigns=int(df.campaign.nunique()),
                n_stable_samples=int(df.n_samples_stable.sum()), lambda_mean=float(lm.mean()),
                lambda_sd=float(lm.std(ddof=1)), cv_pct=float(lm.std(ddof=1) / lm.mean() * 100))


## Figure

In [ ]:
import json
from IPython.display import Image, display

ns = main()
s = summary(ns)
(PROCESSED / 'figS06_summary.json').write_text(json.dumps(s, indent=1))
print(json.dumps(s, indent=1))


In [ ]:
display(Image(str(OUT / 'reproducibility_interexperiment.png')))
display(Image(str(OUT / 'lambda_max_by_campaign.png')))
